In [6]:
import pandas as pd
import json
import os


# Collect all JSON data, split by model name
root_dir = "../export"
skip_files = [
    "characters.json",
    "schema.json",
]
skip_models = [
    "gemini-2.0-flash-old",
    "2.5-pro-exp-unstructured",
]

records = []

for dirpath, _, filenames in os.walk(root_dir):
    for filename in filenames:
        if not filename.endswith(".json") or filename in skip_files:
            continue
        if "pkna-" not in dirpath:
            continue

        generation = dirpath.split("/")[-1]
        pknum = dirpath.split("/")[-2]

        if generation in skip_models:
            continue
        
        filepath = os.path.join(dirpath, filename)
        with open(filepath, "r") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON from {filepath}: {e}")
                continue

        # Determine model name and prompt version
        meta = data.get("metadata", {})
        model = meta.get("model_name")
        version = meta.get("prompt_version")

        records.append({
            "pknum": pknum,
            "generation": generation,
            "data": data,
            "model": model,
            "version": version,
        })

df = pd.DataFrame.from_records(records)
df.head()

,pknum,generation,data,model,version
0,pkna-2,gemini-2.5-pro-exp-03-25,"{'scenes': [{'frames': [{'page': 23, 'frame': ...",None,None
1,pkna-2,gemini-2.5-pro-exp-03-25,"{'scenes': [{'frames': [{'page': 13, 'frame': ...",None,None
2,pkna-2,gemini-2.5-pro-exp-03-25,"{'scenes': [{'frames': [{'page': 3, 'frame': 1...",None,None
3,pkna-2,gemini-2.5-pro-exp-03-25,"{'scenes': [{'frames': [{'page': 63, 'frame': ...",None,None
4,pkna-2,gemini-2.5-pro-exp-03-25,"{'scenes': [{'frames': [{'page': 53, 'frame': ...",None,None


In [9]:
# Group by generation and pknum
grouped = df[["pknum", "generation"]].groupby(["pknum"])
gens = grouped.agg(lambda x: list(set(x)))
gens

,generation
pknum,
pkna-0,[gemini-2.5-pro-exp-03-25]
pkna-0-2,[gemini-2.5-pro-exp-03-25]
pkna-0-3,"[gemini-2.5-pro-exp-03-25, gemini-2.0-flash]"
pkna-1,[gemini-2.5-pro-exp-03-25]
pkna-10,[gemini-2.5-pro-exp-03-25]
pkna-11,"[gemini-2.0-flash, 2.5-pro-exp]"
pkna-12,[gemini-2.5-pro-exp-03-25]
pkna-13,[gemini-2.0-flash]
pkna-14,[gemini-2.0-flash]


In [10]:
# Extract all characters from all scenes
from collections import defaultdict

characters = defaultdict(int)

for _, row in df.iterrows():
    data = row["data"]

    for scene in data["scenes"]:
        for frame in scene["frames"]:
            for bubble in frame["bubbles"]:
                if "character" not in bubble:
                    continue
                c = str(bubble["character"])
                characters[c.lower()] += 1

len(characters)

1954

In [11]:
# Sort characters by number of appearances
sorted_characters = sorted(characters.items(), key=lambda x: x[1], reverse=True)
sorted_characters[:20]

[('paperinik', 5262),
 ('uno', 2122),
 ('paperino', 1636),
 ('angus fangus', 1131),
 ('lyla lay', 1070),
 ('razziatore', 760),
 ('evroniano', 670),
 ('zoster', 612),
 ('xadhoom', 463),
 ('due', 252),
 ('mago', 243),
 ('odin eidolon', 219),
 ('everett ducklair', 216),
 ('zotnam', 208),
 ('zondag', 182),
 ('paperone', 176),
 ('evroniano 1', 156),
 ('sconosciuto', 146),
 ('camera 9', 141),
 ('personaggio non identificato', 128)]